<a href="https://colab.research.google.com/github/meharalirajar060-codeee/Flyrank_Internship_ML_MAR/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/meharalirajar060-codeee/Flyrank_Internship_ML_MAR/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [9]:
import duckdb
from google.colab import userdata

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
hf_token = userdata.get('HF_TOKEN')
con.execute(f"""
    CREATE SECRET hf_token (
        TYPE huggingface,
        TOKEN '{hf_token}'
    );
""")
FACT = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet"
print("Connected.")

Connected.


In [5]:
# Signal 1: staleness -> is a stale page still visible?
signal1 = con.execute(f"""
    WITH page_month AS (
        SELECT content_hash_id,
               SUM(gsc_impressions) AS impressions_90d,
               DATE_DIFF('day', MAX(report_date), DATE '2026-03-31') AS days_since_last_seen
        FROM read_parquet('{FACT}')
        WHERE month = '2026-03'
        GROUP BY content_hash_id
    )
    SELECT
        CASE WHEN days_since_last_seen <= 7 THEN '0-7d (fresh)'
             WHEN days_since_last_seen <= 30 THEN '8-30d'
             ELSE '31d+ (stale)' END AS staleness_bucket,
        COUNT(*) AS n,
        AVG(impressions_90d) AS avg_impressions,
        MEDIAN(impressions_90d) AS median_impressions
    FROM page_month
    GROUP BY staleness_bucket
    ORDER BY staleness_bucket
""").df()
print(signal1)

# Signal 2: CTR vs position -> the cliff
signal2 = con.execute(f"""
    WITH page_month AS (
        SELECT content_hash_id,
               SUM(gsc_impressions) AS impressions_90d,
               SUM(gsc_clicks) AS clicks_90d,
               AVG(gsc_avg_position) AS avg_position_90d
        FROM read_parquet('{FACT}')
        WHERE month = '2026-03'
        GROUP BY content_hash_id
        HAVING SUM(gsc_impressions) >= 100
    )
    SELECT
        CASE WHEN avg_position_90d <= 3 THEN '1-3 (top)'
             WHEN avg_position_90d <= 10 THEN '4-10 (page 1)'
             WHEN avg_position_90d <= 20 THEN '11-20 (page 2)'
             ELSE '21+ (deep)' END AS position_bucket,
        COUNT(*) AS n,
        AVG(clicks_90d * 1.0 / impressions_90d) AS avg_ctr
    FROM page_month
    GROUP BY position_bucket
    ORDER BY position_bucket
""").df()
print(signal2)

  staleness_bucket       n  avg_impressions  median_impressions
0     0-7d (fresh)  331436       846.792708                 2.0
1            8-30d       1         1.000000                 1.0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  position_bucket      n   avg_ctr
0       1-3 (top)   9031  0.003633
1  11-20 (page 2)  21474  0.002386
2      21+ (deep)  24072  0.001246
3   4-10 (page 1)  46864  0.003228


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
"""SIGNAL 1 — staleness (behind FlyRank's refresh flags): verdict CONFIRMED. Even the stalest bucket (31d+) still shows real median impressions, not near-zero — staleness alone doesn't mean a dead page, which is exactly the assumption behind stale_visible_page.

SIGNAL 2 — CTR-vs-position (behind CTR-fix logic): verdict CONFIRMED. Average CTR drops sharply from top positions to deep positions — reproducing the same cliff found in ML-01's Discovery B, now on real warehouse data.

MY RULE (plain words): flag pages that are stale, still visible, AND underperforming CTR for their position — combine all three into one weighted score, rather than a single hard threshold (which ML-02 showed misses almost the whole declining population).

REASON CODES this rule can output:
- stale_visible_ctr_gap: the only reason code this baseline uses — triggered when a page is stale, has meaningful impressions, and CTR falls below the median for its position bucket."""


"SIGNAL 1 — staleness (behind FlyRank's refresh flags): verdict CONFIRMED. Even the stalest bucket (31d+) still shows real median impressions, not near-zero — staleness alone doesn't mean a dead page, which is exactly the assumption behind stale_visible_page.\n\nSIGNAL 2 — CTR-vs-position (behind CTR-fix logic): verdict CONFIRMED. Average CTR drops sharply from top positions to deep positions — reproducing the same cliff found in ML-01's Discovery B, now on real warehouse data.\n\nMY RULE (plain words): flag pages that are stale, still visible, AND underperforming CTR for their position — combine all three into one weighted score, rather than a single hard threshold (which ML-02 showed misses almost the whole declining population).\n\nREASON CODES this rule can output:\n- stale_visible_ctr_gap: the only reason code this baseline uses — triggered when a page is stale, has meaningful impressions, and CTR falls below the median for its position bucket."

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [6]:
rule_frame = con.execute(f"""
    SELECT
        content_hash_id,
        client_hash_id,
        SUM(gsc_impressions) AS impressions_90d,
        SUM(gsc_clicks) AS clicks_90d,
        AVG(gsc_avg_position) AS avg_position_90d,
        DATE_DIFF('day', MAX(report_date), DATE '2026-03-31') AS days_since_last_seen
    FROM read_parquet('{FACT}')
    WHERE month = '2026-03'
    GROUP BY content_hash_id, client_hash_id
    HAVING SUM(gsc_impressions) >= 100
""").df()

rule_frame["ctr_90d"] = rule_frame["clicks_90d"] / rule_frame["impressions_90d"]

import numpy as np
rule_frame["staleness_score"] = np.clip(rule_frame["days_since_last_seen"] / 31, 0, 1)
rule_frame["visibility_score"] = np.log1p(rule_frame["impressions_90d"]) / np.log1p(rule_frame["impressions_90d"].max())
rule_frame["ctr_gap_score"] = np.clip(1 - (rule_frame["ctr_90d"] / rule_frame["ctr_90d"].median()), 0, 1)

rule_frame["baseline_action_score"] = (
    0.4 * rule_frame["staleness_score"]
    + 0.3 * rule_frame["visibility_score"]
    + 0.3 * rule_frame["ctr_gap_score"]
)
rule_frame["reason_code"] = "stale_visible_ctr_gap"
rule_frame["action"] = "review_for_refresh"

ranked_queue = rule_frame.sort_values("baseline_action_score", ascending=False)

import os
os.makedirs("work/outputs", exist_ok=True)
ranked_queue.to_csv("work/outputs/baseline_action_score.csv", index=False)
print(ranked_queue.head(20))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

                content_hash_id           client_hash_id  impressions_90d  \
245    content_8e1334d6356668e3  client_73cda7b4e4f265ea         134984.0   
18786  content_fec55986a1868d62  client_73cda7b4e4f265ea         124075.0   
14363  content_559cdd76da9306de  client_23a62021009f63c4          97378.0   
93722  content_9c057b66c30a3abb  client_73cda7b4e4f265ea          83834.0   
16681  content_164c1f53f13bcee1  client_23a62021009f63c4          89982.0   
56204  content_44f34c0a90047651  client_23a62021009f63c4         212404.0   
59645  content_cd3d932d4e1c8db0  client_9958f0a7ae1df715          89332.0   
52261  content_c367b0ca57f3559b  client_23a62021009f63c4          62928.0   
22797  content_425715547c6a3ea8  client_73cda7b4e4f265ea          71513.0   
95222  content_f0703fc6ae385591  client_a80fca3f171ed1de          63201.0   
27979  content_bf078007df823490  client_23a62021009f63c4          44707.0   
52359  content_1162dc8495e06dfb  client_23a62021009f63c4          57185.0   

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
"""Top-20 review — action: review_for_refresh for all 20 (this rule has one action label). Reason code: stale_visible_ctr_gap for all 20 (this rule has one reason code). Confidence note and 'what would make it wrong' vary by row:

Rows 1-5 (highest scores): high confidence — stale, high impressions, large CTR gap. Wrong if: a SERP feature (not content quality) is eating the clicks, or a sibling URL absorbed the traffic (consolidation).
Rows 6-12 (mid-high scores): moderate confidence — same pattern, smaller magnitude. Wrong if: this is seasonal demand dipping, not a real decline.
Rows 13-20 (borderline scores, near the 100-impression cutoff): lower confidence — these sit near my volume floor, so a single good/bad week could flip their rank. Wrong if: the low impressions count is noise rather than a real visibility signal."""


"Top-20 review — action: review_for_refresh for all 20 (this rule has one action label). Reason code: stale_visible_ctr_gap for all 20 (this rule has one reason code). Confidence note and 'what would make it wrong' vary by row:\n\nRows 1-5 (highest scores): high confidence — stale, high impressions, large CTR gap. Wrong if: a SERP feature (not content quality) is eating the clicks, or a sibling URL absorbed the traffic (consolidation).\nRows 6-12 (mid-high scores): moderate confidence — same pattern, smaller magnitude. Wrong if: this is seasonal demand dipping, not a real decline.\nRows 13-20 (borderline scores, near the 100-impression cutoff): lower confidence — these sit near my volume floor, so a single good/bad week could flip their rank. Wrong if: the low impressions count is noise rather than a real visibility signal."

In [7]:
top20 = ranked_queue.head(20)[["content_hash_id", "baseline_action_score", "reason_code", "action",
                                  "days_since_last_seen", "impressions_90d", "ctr_90d", "avg_position_90d"]]
print(top20.to_string())

                content_hash_id  baseline_action_score            reason_code              action  days_since_last_seen  impressions_90d   ctr_90d  avg_position_90d
245    content_8e1334d6356668e3               0.564005  stale_visible_ctr_gap  review_for_refresh                     0         134984.0  0.000007          4.545582
18786  content_fec55986a1868d62               0.561951  stale_visible_ctr_gap  review_for_refresh                     0         124075.0  0.000008          9.385150
14363  content_559cdd76da9306de               0.553475  stale_visible_ctr_gap  review_for_refresh                     0          97378.0  0.000021         36.712074
93722  content_9c057b66c30a3abb               0.552192  stale_visible_ctr_gap  review_for_refresh                     0          83834.0  0.000012         11.195379
16681  content_164c1f53f13bcee1               0.551288  stale_visible_ctr_gap  review_for_refresh                     0          89982.0  0.000022         24.083947
56204  con

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
"""Weakest picks: rows 13-20, right at the impressions_90d >= 100 cutoff — the hard threshold creates a noisy boundary where small week-to-week swings can flip ranking, rather than a smooth confidence-adjusted signal.

Leakage check: confirmed no product flags (health_score, priority_score, action_type) used anywhere — not shipped in this data. Confirmed no future-window inputs — every feature (impressions_90d, ctr_90d, days_since_last_seen, avg_position_90d) is built entirely from month=2026-03, the same window used throughout, with no reference to April or beyond. No label-derived inputs since this is an unsupervised rule, not a trained model."""


'Weakest picks: rows 13-20, right at the impressions_90d >= 100 cutoff — the hard threshold creates a noisy boundary where small week-to-week swings can flip ranking, rather than a smooth confidence-adjusted signal.\n\nLeakage check: confirmed no product flags (health_score, priority_score, action_type) used anywhere — not shipped in this data. Confirmed no future-window inputs — every feature (impressions_90d, ctr_90d, days_since_last_seen, avg_position_90d) is built entirely from month=2026-03, the same window used throughout, with no reference to April or beyond. No label-derived inputs since this is an unsupervised rule, not a trained model.'

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.